## Notebook39

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! python -m spacy download en_core_web_sm

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs
from funs import DSText

import spacy

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
docs_raw = (
    pl.read_parquet(ub + "data/imdb5k_pca.parquet")
    .select(c.id, c.label, c.text)
    .rename({"id": "doc_id"})
    .sort(c.doc_id)
    .group_by(c.label, maintain_order=True)
    .head(100)
)

In [ ]:
nlp = spacy.load("en_core_web_sm")

### Questions

`docs_raw` holds 200 IMDb movie reviews, 100 labeled `"positive"` and 100
labeled `"negative"`, each with a `doc_id` and the raw `text` of the review.
This is a smaller corpus than the Wikipedia author pages the chapter uses,
and it has a mess of its own. The reviews were scraped from web pages, and
many still contain the literal characters `<br />` (an HTML line break). The
first few questions deal with that mess directly, in the spirit of the
chapter's warning that the wrapper functions are not magic. Print results
unless a question says otherwise.

1. In the first block, count how many reviews fall into each `label`. In the
second block, count how many of the 200 reviews contain the literal string
`"<br />"` somewhere in their `text`.

In [ ]:
(
    docs_raw
    .group_by(c.label)
    .agg(n = pl.len())
)

In [ ]:
(
    docs_raw
    .select(c.text.str.contains("<br />").sum())
)

2. Before cleaning anything, see what the raw HTML does to the pipeline.
Run `DSText.process()` on just review `doc0001`, then filter the result to
tokens whose `token` contains `"<br"` and print `token`, `upos`, and
`is_alpha`.

In [ ]:
example = DSText.process(
    docs_raw.filter(c.doc_id == "doc0001").select(c.doc_id, c.text),
    nlp
)

(
    example
    .filter(c.token.str.contains("<br"))
    .select(c.token, c.upos, c.is_alpha)
)

Every one of these tokens has `is_alpha` equal to `False`, but four of them
still get tagged `NOUN`, and the same string, `"/><br"`, comes back tagged
three different ways (`PUNCT`, `NOUN`, `NUM`) depending on its neighbors.
The tagger is being handed text that was never a real sentence. The
chapter's noun-frequency query in the next section filters only on `upos`,
so these tokens would get into a noun count because `is_alpha` is never
checked.

3. Build `docs` from `docs_raw` by replacing every `"<br />"` with a single
space. Then run `DSText.process()` on the full, cleaned `docs` table to build
`anno` and print its shape.

In [ ]:
docs = (
    docs_raw
    .with_columns(
        text = c.text.str.replace_all("<br />", " ")
    )
)

anno = DSText.process(docs.select(c.doc_id, c.text), nlp)
anno.shape

4. Look at the first sentence of `doc0001` on its own: filter `anno` to
`doc_id == "doc0001"` and `sid == 1`, and select `tid`, `token`, and `upos`.

In [ ]:
(
    anno
    .filter(c.doc_id == "doc0001", c.sid == 1)
    .select(c.tid, c.token, c.upos)
)

5. In the first block, find the 10 most frequent nouns across the whole
corpus: filter `anno` to `upos == "NOUN"`, group by `lemma`, count, and sort
in descending order. In the second block, plot them as a bar chart with
`geom_col()`, flipping the coordinates so the noun labels are readable.

In [ ]:
top_nouns = (
    anno
    .filter(c.upos == "NOUN")
    .group_by(c.lemma)
    .agg(count = pl.len())
    .sort(c.count, descending=True)
    .head(10)
)
top_nouns

In [ ]:
(
    top_nouns
    .ggplot(aes(c.lemma, c.count))
    .geom_col()
    .coord_flip()
    .labs(title="Most Frequent Nouns", x="Noun", y="Count")
)

In the chapter's biography pages the most common nouns spread across a
handful of themes. This list is dominated by the two words for the thing
being reviewed: `movie` and `film` together account for more mentions than
the next four nouns combined. The most frequent word in a collection is not
always the most informative one.

6. In the first block, find the 10 most frequent adjectives within each
`label`. Join `anno` to the `doc_id`/`label` pairs in `docs`, filter to
alphabetic adjectives, group by `label` and `lemma`, count, and take the top
10 rows within each `label`. In the second block, filter `adj_by_label` to
just the lemmas `"good"` and `"bad"` and print the count for each label.
Sentiment labels aside, why doesn't a simple count of these two words cleanly
separate the positive reviews from the negative ones?

In [ ]:
adj_by_label = (
    anno
    .join(docs.select(c.doc_id, c.label), on=c.doc_id)
    .filter(c.upos == "ADJ", c.is_alpha)
    .group_by(c.label, c.lemma)
    .agg(count = pl.len())
    .sort([c.label, c.count], descending=[False, True])
)

(
    adj_by_label
    .group_by(c.label, maintain_order=True)
    .head(10)
)

In [ ]:
(
    adj_by_label
    .filter(c.lemma.is_in(["good", "bad"]))
    .sort(c.label, c.lemma)
)

**Answer**:
